In [4]:
# ============================================
# Add Sentiment Score to Stock Data (FIXED)
# Author: Samaneh Kavianfar
# ============================================

import yfinance as yf
import feedparser
import pandas as pd
from nltk.sentiment import SentimentIntensityAnalyzer
from sqlalchemy import create_engine
from sqlalchemy import URL
from urllib.parse import quote

import nltk
import time
import requests
from bs4 import BeautifulSoup

# Download VADER lexicon (run once)
nltk.download('vader_lexicon')

# ============================================
# Connect to PostgreSQL
# ============================================

url = URL.create(
    drivername='postgresql',
    username='postgres',
    password = '@Taha1391',
    host='localhost',
    port=5432,
    database='stock_db'
)

engine = create_engine(url)
print("✅ Connected to PostgreSQL!")

# ============================================
# Function to fetch news and calculate sentiment
# ============================================
def get_sentiment_score(symbol):
    """Fetches news for a symbol and returns average VADER sentiment score."""
    sia = SentimentIntensityAnalyzer()
    total_score = 0
    count = 0

    print(f" 📰 Fetching news for {symbol}...")

    # 1. Try Yahoo Finance news
    ticker = yf.Ticker(symbol)
    news = ticker.news

    if news:
        print(f" ✅ Found {len(news)} news from Yahoo Finance")
        for item in news[:10]:
            headline = item.get('title', '')
            if headline:
                sentiment = sia.polarity_scores(headline)
                total_score += sentiment['compound']
                count += 1

    # 2. If no news, try Google News RSS
    if count == 0:
        print(f" ⚠️ No news from Yahoo, trying Google News RSS...")
        try:
            feed_url = f"https://news.google.com/rss/search?q={symbol}+stock&hl=en-US&gl=US&ceid=US:en"
            feed = feedparser.parse(feed_url)
            if feed.entries:
                print(f" ✅ Found {len(feed.entries)} news from Google News")
                for entry in feed.entries[:10]:
                    headline = entry.title
                    sentiment = sia.polarity_scores(headline)
                    total_score += sentiment['compound']
                    count += 1
        except Exception as e:
            print(f" ⚠️ Google News RSS failed: {e}")

    # 3. If still no news, try a simple fallback (use stock price change as proxy)
    if count == 0:
        print(f" ⚠️ No news found for {symbol}, using price change as proxy")
        try:
            ticker = yf.Ticker(symbol)
            hist = ticker.history(period="5d")
            if not hist.empty:
                price_change = hist['Close'].pct_change().iloc[-1]
                # Convert price change to a sentiment score (-1 to +1)
                sentiment_score = max(-1, min(1, price_change * 5))
                return sentiment_score
        except:
            return 0.0

    if count == 0:
        return 0.0

    return total_score 

# ============================================
# Get list of symbols from database
# ============================================
df_symbols = pd.read_sql("SELECT DISTINCT symbol FROM fct_daily_returns", engine)
symbols = df_symbols['symbol'].tolist()
print(f"📊 Analyzing sentiment for {len(symbols)} symbols...")

# ============================================
# Process each symbol
# ============================================
sentiment_data = []
for symbol in symbols:
    sentiment = get_sentiment_score(symbol)
    sentiment_data.append({
        'symbol': symbol,
        'sentiment_score': sentiment,
    })
    time.sleep(0.5) # Be polite to the API

# ============================================
# Save results to PostgreSQL
# ============================================
sentiment_df = pd.DataFrame(sentiment_data)
sentiment_df.to_sql('stock_sentiments', engine, if_exists='append', index=False)
print(f"✅ Sentiment scores saved to 'stock_sentiments' table!")

# ============================================
# Show results
# ============================================
print("\n📊 Sentiment scores:")
print(sentiment_df)

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Toranj\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


✅ Connected to PostgreSQL!
📊 Analyzing sentiment for 9 symbols...
 📰 Fetching news for AAPL...
 ✅ Found 10 news from Yahoo Finance
 ⚠️ No news from Yahoo, trying Google News RSS...
 ✅ Found 100 news from Google News
 📰 Fetching news for MSFT...
 ✅ Found 10 news from Yahoo Finance
 ⚠️ No news from Yahoo, trying Google News RSS...
 ✅ Found 100 news from Google News
 📰 Fetching news for GOOGL...
 ✅ Found 10 news from Yahoo Finance
 ⚠️ No news from Yahoo, trying Google News RSS...
 ✅ Found 100 news from Google News
 📰 Fetching news for JPM...
 ✅ Found 10 news from Yahoo Finance
 ⚠️ No news from Yahoo, trying Google News RSS...
 ✅ Found 100 news from Google News
 📰 Fetching news for NKE...
 ✅ Found 10 news from Yahoo Finance
 ⚠️ No news from Yahoo, trying Google News RSS...
 ✅ Found 100 news from Google News
 📰 Fetching news for AMZN...
 ✅ Found 10 news from Yahoo Finance
 ⚠️ No news from Yahoo, trying Google News RSS...
 ✅ Found 100 news from Google News
 📰 Fetching news for VTI...
 ✅ Foun